ComfyUI Template

In [ ]:
#@title Thiết lập môi trường
import subprocess
import requests
from pathlib import Path

OPTIONS = {}

#USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
#UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = 'ComfyUI'
#OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
#OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

#if OPTIONS['USE_GOOGLE_DRIVE']:
#    !echo "Mounting Google Drive..."
#    %cd /

#    from google.colab import drive
#    drive.mount('/content/drive')

#    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
#    %cd /content/drive/MyDrive

![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

!echo -= Install dependencies =-
!pip install xformers -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
!apt -y install -qq aria2

In [ ]:
#@title Cài đặt custom nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager
#

#
!find . -name "requirements.txt" -exec pip install -r {} \;

In [ ]:
#@title Tải model
# !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/SporkySporkness/FLUX.1-Fill-dev-GGUF/resolve/main/flux1-fill-dev-fp16-Q8_0-GGUF.gguf -d /content/ComfyUI/models/unet -o "flux1-fill-dev-fp16-Q8_0-GGUF.gguf"

### Chạy ComfyUI sử dụng cloudflared




In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server